# Queensland Spatial SQL Demo — Read `.tab` → Spatial SQL → Write `.tab`

End-to-end geospatial round-trip on **serverless** compute, using a custom PySpark
data source and native Databricks **Spatial SQL**:

1. **Source** — download real **Queensland** boundaries (ABS ASGS 2021 **SA2** for
   Greater Brisbane) from the official ABS ArcGIS service and seed a **MapInfo `.tab`**
   file in a Unity Catalog Volume.
2. **Read** — load the `.tab` with the custom `geo` data source (`spark.read.format("geo")`).
3. **Process** — native Spatial SQL: reproject with `ST_Transform` and compute polygon
   area with `ST_Area` (validated against ABS's own published area figures).
4. **Write** — persist the enriched result **back to a `.tab`** with the custom
   `geo` writer (`df.write.format("geo")`).
5. **Verify** — read the output `.tab` back and confirm feature count + new column.

This notebook is **self-contained**: it defines the `geo` data source inline (no
`%run`), installs its dependencies, and creates its own catalog/schema/volume. Run
top to bottom from a cold start.

> **⚠️ Serverless requirement — Environment version 2+.** Custom PySpark data
> sources (`spark.dataSource.register`) need **serverless environment version 2 or
> higher**. On an interactive serverless notebook, open the **Environment** side
> panel (top-right), set **Version** to `2` (or later), and Apply. If you see
> `AttributeError: 'SparkSession' object has no attribute 'dataSource'`, the
> environment version is too low. (When run as a Job, set the task's environment
> spec `client` to `"2"`.)

> **Why seed the `.tab`?** The ABS retired MapInfo TAB downloads (SA1/SA2 now ship as
> Shapefile / GeoPackage), and Queensland's QSpatial `.tab` files sit behind an
> interactive order flow with no stable URL. So we pull authoritative ABS geometry
> from the ArcGIS service and write it to a `.tab` first — exactly the file a customer
> would have exported from MapInfo — then read it back through the custom source.

## 1. Setup — install geospatial libraries

Serverless does not ship Fiona/Shapely/pyproj, so install them. `fiona` provides the
MapInfo `.tab` driver; `shapely`/`pyproj` handle geometry and CRS on the driver.

In [ ]:
%pip install fiona shapely pyproj --quiet

In [ ]:
dbutils.library.restartPython()

## 2. Configure Unity Catalog location

Widgets so this is portable across workspaces — set catalog / schema / volume once.

In [ ]:
dbutils.widgets.text("catalog_name", "danny_catalog", "Catalog Name")
dbutils.widgets.text("schema_name", "qld_spatial_demo", "Schema Name")
dbutils.widgets.text("volume_name", "geo", "Volume Name")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")

# The catalog is assumed to already exist. We deliberately do NOT run CREATE CATALOG:
# on metastores with Default Storage enabled, CREATE CATALOG requires an explicit
# MANAGED LOCATION and errors otherwise, even with IF NOT EXISTS. The schema and
# volume inherit the catalog's storage, so creating just those is safe and portable.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
print(f"Catalog : {catalog_name}")
print(f"Schema  : {catalog_name}.{schema_name}")
print(f"Volume  : {volume_path}")

## 3. Define the `geo` custom data source (read + write) inline

The full source lives in `01_geo_datasource_definition.py`; it is inlined here so the
demo runs with no `%run` dependency. It supports reading many geospatial formats and
**writing** them back — including MapInfo `.tab`.

In [ ]:
from pyspark.sql.datasource import (
    DataSource, DataSourceReader, DataSourceWriter, WriterCommitMessage, InputPartition,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, LongType, BooleanType,
)
from dataclasses import dataclass
from typing import Iterator, Tuple, List


class GeoPartition(InputPartition):
    def __init__(self, start_idx: int, end_idx: int):
        self.start_idx = start_idx
        self.end_idx = end_idx


# Map an output file extension to its Fiona/OGR driver.
_EXT_TO_DRIVER = {
    ".tab": "MapInfo File",
    ".geojson": "GeoJSON",
    ".json": "GeoJSON",
    ".shp": "ESRI Shapefile",
    ".gpkg": "GPKG",
    ".kml": "KML",
}


class GeoDataSource(DataSource):
    """Custom PySpark data source for reading and writing geospatial files.

    Read:  spark.read.format("geo").option("path", "...").load()
    Write: df.write.format("geo").option("path", "out.tab").option("crs", "EPSG:4326").save()
    """

    @classmethod
    def name(cls) -> str:
        return "geo"

    def schema(self) -> StructType:
        return self._infer_schema()

    def _infer_schema(self) -> StructType:
        import fiona
        path = self.options.get("path")
        if not path:
            raise ValueError("The 'path' option is required for the geo data source")
        layer = self.options.get("layer")
        with fiona.open(path, layer=layer) as src:
            fiona_schema = src.schema
        fields = [
            StructField("geometry", StringType(), True),
            StructField("geometry_type", StringType(), True),
        ]
        fiona_to_spark = {
            "int": IntegerType(), "int32": IntegerType(), "int64": LongType(),
            "float": DoubleType(), "str": StringType(), "bool": BooleanType(),
        }
        for prop_name, prop_type in fiona_schema["properties"].items():
            base = prop_type.split(":")[0] if ":" in prop_type else prop_type
            fields.append(StructField(prop_name, fiona_to_spark.get(base, StringType()), True))
        return StructType(fields)

    def reader(self, schema: StructType) -> "GeoDataSourceReader":
        return GeoDataSourceReader(schema, self.options)

    def writer(self, schema: StructType, overwrite: bool) -> "GeoDataSourceWriter":
        return GeoDataSourceWriter(schema, self.options, overwrite)


class GeoDataSourceReader(DataSourceReader):
    """Streaming reader using Fiona (memory efficient, chunked partitions)."""

    def __init__(self, schema: StructType, options: dict):
        self.schema = schema
        self.options = options

    def partitions(self) -> List[InputPartition]:
        import fiona
        path = self.options.get("path")
        layer = self.options.get("layer")
        chunk_size = int(self.options.get("chunk_size", 10000))
        with fiona.open(path, layer=layer) as src:
            total = len(src)
        return [GeoPartition(s, min(s + chunk_size, total)) for s in range(0, total, chunk_size)]

    def read(self, partition: "GeoPartition") -> Iterator[Tuple]:
        import fiona, warnings
        from shapely.geometry import shape
        from pyproj import Transformer, CRS
        warnings.filterwarnings("ignore")

        path = self.options.get("path")
        layer = self.options.get("layer")
        geometry_format = self.options.get("geometry_format", "wkt").lower()
        target_crs = self.options.get("crs")
        schema_cols = [f.name for f in self.schema.fields if f.name not in ("geometry", "geometry_type")]

        transformer = None
        with fiona.open(path, layer=layer) as src:
            if target_crs and src.crs:
                try:
                    transformer = Transformer.from_crs(
                        CRS.from_user_input(src.crs), CRS.from_user_input(target_crs), always_xy=True
                    )
                except Exception:
                    transformer = None

        with fiona.open(path, layer=layer) as src:
            for i, feature in enumerate(src):
                if i < partition.start_idx:
                    continue
                if i >= partition.end_idx:
                    break
                values = []
                geom = feature.get("geometry")
                if geom:
                    try:
                        g = shape(geom)
                        if transformer:
                            from shapely.ops import transform
                            g = transform(transformer.transform, g)
                        values.append(g.wkb_hex if geometry_format == "wkb" else g.wkt)
                        values.append(g.geom_type)
                    except Exception:
                        values.extend([None, None])
                else:
                    values.extend([None, None])
                props = feature.get("properties", {})
                for col in schema_cols:
                    val = props.get(col)
                    if val is None or (isinstance(val, float) and val != val):
                        values.append(None)
                    else:
                        values.append(val)
                yield tuple(values)


@dataclass
class GeoCommitMessage(WriterCommitMessage):
    staging_file: str
    count: int


class GeoDataSourceWriter(DataSourceWriter):
    """Writer: executors stage rows as NDJSON; the driver merges into one file.

    A .tab is a single logical dataset (.tab/.dat/.map/.id), not splittable like
    Parquet, so partitions stage portable NDJSON (WKT + attributes) into a shared
    directory and the driver authors one output file in commit().
    """

    def __init__(self, schema: StructType, options: dict, overwrite: bool):
        self.schema = schema
        self.options = options
        self.overwrite = overwrite
        self.path = options.get("path")
        if not self.path:
            raise ValueError("The 'path' option is required to write the geo data source")
        self.crs = options.get("crs", "EPSG:4326")
        self.str_width = int(options.get("str_width", 254))
        self.staging_dir = self.path + "__geo_staging"
        self.attr_cols = [f.name for f in schema.fields if f.name not in ("geometry", "geometry_type")]

    def write(self, iterator) -> GeoCommitMessage:
        import os, json
        from pyspark import TaskContext
        pid = TaskContext.get().partitionId()
        os.makedirs(self.staging_dir, exist_ok=True)
        staging_file = os.path.join(self.staging_dir, f"part-{pid:05d}.ndjson")
        count = 0
        with open(staging_file, "w") as out:
            for row in iterator:
                d = row.asDict()
                out.write(json.dumps(
                    {"wkt": d.get("geometry"), "props": {c: d.get(c) for c in self.attr_cols}},
                    default=str,
                ) + "\n")
                count += 1
        return GeoCommitMessage(staging_file=staging_file, count=count)

    def commit(self, messages) -> None:
        import os, glob, json, shutil, tempfile, fiona
        from shapely import wkt as shapely_wkt
        from shapely.geometry import mapping

        ext = os.path.splitext(self.path)[1].lower()
        driver = _EXT_TO_DRIVER.get(ext, "MapInfo File")

        spark_to_fiona = {"integer": "int32", "long": "int64", "double": "float",
                          "float": "float", "boolean": "bool"}
        prop_schema = {}
        for f in self.schema.fields:
            if f.name in ("geometry", "geometry_type"):
                continue
            base = f.dataType.typeName()
            prop_schema[f.name] = f"str:{self.str_width}" if base == "string" \
                else spark_to_fiona.get(base, f"str:{self.str_width}")
        out_schema = {"geometry": "Unknown", "properties": prop_schema}

        # Author on a local temp dir (seek-safe), then copy sidecars to the Volume.
        # tempfile.mkdtemp() resolves to a writable dir; on serverless the data-source
        # sandbox has /tmp writable but /local_disk0 read-only.
        base_name = os.path.splitext(os.path.basename(self.path))[0]
        local_dir = tempfile.mkdtemp(prefix="geo_write_")
        local_path = os.path.join(local_dir, base_name + ext)

        total = 0
        with fiona.open(local_path, "w", driver=driver, crs=self.crs, schema=out_schema) as dst:
            for msg in messages:
                if msg is None:
                    continue
                with open(msg.staging_file) as sf:
                    for line in sf:
                        rec = json.loads(line)
                        if not rec.get("wkt"):
                            continue
                        dst.write({"geometry": mapping(shapely_wkt.loads(rec["wkt"])),
                                   "properties": rec["props"]})
                        total += 1

        target_dir = os.path.dirname(self.path)
        if target_dir:
            os.makedirs(target_dir, exist_ok=True)
        for produced in glob.glob(os.path.join(local_dir, base_name + ".*")):
            dst_file = os.path.join(target_dir, os.path.basename(produced))
            if self.overwrite and os.path.exists(dst_file):
                try:
                    os.remove(dst_file)
                except OSError:
                    pass
            # copyfile (bytes only) — NOT copy2/copy: UC Volume FUSE rejects the
            # utime() metadata-copy those do (PermissionError [Errno 1]).
            shutil.copyfile(produced, dst_file)

        shutil.rmtree(local_dir, ignore_errors=True)
        shutil.rmtree(self.staging_dir, ignore_errors=True)
        print(f"[geo writer] wrote {total} features to {self.path} (driver='{driver}', crs='{self.crs}')")

    def abort(self, messages) -> None:
        import shutil
        shutil.rmtree(self.staging_dir, ignore_errors=True)


spark.dataSource.register(GeoDataSource)
print("Registered custom data source: 'geo' (read + write)")

## 4. Source data — seed a Queensland `.tab` from ABS ASGS 2021 SA2

Pull **Greater Brisbane SA2 polygons** from the official ABS ArcGIS service as GeoJSON,
then write them to a MapInfo `.tab` in the Volume. This is the input `.tab` a customer
would bring — here it is sourced from authoritative ABS geometry so the demo is
fully reproducible.

In [ ]:
import json, urllib.parse, urllib.request, os

source_tab = f"{volume_path}/qld_sa2_greater_brisbane.tab"

# ABS ASGS 2021 SA2 boundaries — official ArcGIS feature service (SA2 layer 0).
ABS_SA2_QUERY = (
    "https://geo.abs.gov.au/arcgis/rest/services/ASGS2021/SA2/MapServer/0/query"
)
params = {
    "where": "state_name_2021='Queensland' AND gccsa_name_2021='Greater Brisbane'",
    "outFields": "sa2_code_2021,sa2_name_2021,sa3_name_2021,sa4_name_2021,gccsa_name_2021,state_name_2021",
    "returnGeometry": "true",
    "outSR": "4326",   # WGS84 lon/lat
    "f": "geojson",
}
url = ABS_SA2_QUERY + "?" + urllib.parse.urlencode(params)

print("Downloading QLD SA2 boundaries from the ABS ArcGIS service...")
req = urllib.request.Request(url, headers={"User-Agent": "databricks-demo"})
with urllib.request.urlopen(req, timeout=90) as resp:
    gj = json.load(resp)

features = gj.get("features", [])
print(f"Downloaded {len(features)} SA2 features")

# Write the GeoJSON to a .tab using the same Fiona MapInfo driver the source expects.
import fiona, warnings
from shapely.geometry import shape, mapping
warnings.filterwarnings("ignore")

prop_schema = {
    "sa2_code_2021": "str:16", "sa2_name_2021": "str:100",
    "sa3_name_2021": "str:100", "sa4_name_2021": "str:100",
    "gccsa_name_2021": "str:100", "state_name_2021": "str:50",
}
seed_schema = {"geometry": "Unknown", "properties": prop_schema}

# Author locally then copy to the Volume (seek-safe for the .map index).
import tempfile, glob, shutil
_seed_local = tempfile.mkdtemp(prefix="seed_")
_seed_path = os.path.join(_seed_local, "qld_sa2_greater_brisbane.tab")
with fiona.open(_seed_path, "w", driver="MapInfo File", crs="EPSG:4326", schema=seed_schema) as dst:
    for f in features:
        props = {k: f["properties"].get(k) for k in prop_schema}
        dst.write({"geometry": f["geometry"], "properties": props})
os.makedirs(volume_path, exist_ok=True)
for produced in glob.glob(os.path.join(_seed_local, "qld_sa2_greater_brisbane.*")):
    # copyfile (bytes only): UC Volume FUSE rejects the utime() metadata-copy in copy2.
    shutil.copyfile(produced, os.path.join(volume_path, os.path.basename(produced)))
shutil.rmtree(_seed_local, ignore_errors=True)

print(f"Seeded source .tab: {source_tab}")
print("Sidecar files in volume:")
for f in sorted(os.listdir(volume_path)):
    if f.startswith("qld_sa2_greater_brisbane"):
        print("  ", f)

## 5. Read the `.tab` with the custom `geo` data source

`spark.read.format("geo")` reads the MapInfo `.tab` and returns geometry as WKT plus
all attribute columns.

In [ ]:
df_sa2 = (
    spark.read.format("geo")
    .option("path", source_tab)
    .load()
)

print("Schema of the .tab read via the custom source:")
df_sa2.printSchema()
print(f"Feature count: {df_sa2.count()}")
display(df_sa2.select("geometry_type", "sa2_code_2021", "sa2_name_2021", "sa3_name_2021", "sa4_name_2021").limit(10))

## 6. Process with native Databricks Spatial SQL

Register the DataFrame as a view and run **Spatial SQL**:
- `ST_GeomFromWKT(geometry, 4326)` — parse WKT to a `GEOMETRY` (WGS84).
- `ST_Transform(..., 3577)` — reproject to **GDA94 / Australian Albers (EPSG:3577)**,
  the standard **equal-area** projection for Australia (in metres) — the correct
  choice for computing area. It covers all of Queensland in one CRS; the MGA
  grid (zones 54–56 across QLD) is *conformal*, so it distorts area and is
  zoned. For strict GDA2020 datum-consistency with the 2021 ABS data use
  **EPSG:9473** (GDA2020 Albers) — the area result is identical to 3577.
- `ST_Area(...) / 1e6` — polygon area in **km²**.
- `ST_Centroid(...)` — representative point (lon/lat) for each SA2.

In [ ]:
df_sa2.createOrReplaceTempView("qld_sa2")

df_enriched = spark.sql("""
    SELECT
        sa2_code_2021,
        sa2_name_2021,
        sa3_name_2021,
        sa4_name_2021,
        gccsa_name_2021,
        state_name_2021,
        geometry,
        -- Area in km² via equal-area Australian Albers (EPSG:3577)
        ROUND(
            ST_Area(ST_Transform(ST_GeomFromWKT(geometry, 4326), 3577)) / 1e6,
            4
        ) AS area_sqkm,
        -- Representative centroid (WGS84 lon/lat)
        ROUND(ST_X(ST_Centroid(ST_GeomFromWKT(geometry, 4326))), 6) AS centroid_lon,
        ROUND(ST_Y(ST_Centroid(ST_GeomFromWKT(geometry, 4326))), 6) AS centroid_lat
    FROM qld_sa2
""")

print("Enriched with ST_Transform + ST_Area (area_sqkm) and ST_Centroid:")
display(
    df_enriched
    .select("sa2_name_2021", "sa4_name_2021", "area_sqkm", "centroid_lon", "centroid_lat")
    .orderBy("area_sqkm", ascending=False)
    .limit(10)
)

### Sanity check — total area of Greater Brisbane
The SA2s should sum to roughly the Greater Brisbane GCCSA footprint (~15,800 km²),
broken down by SA4 region.

In [ ]:
df_enriched.createOrReplaceTempView("qld_sa2_enriched")

display(spark.sql("""
    SELECT
        sa4_name_2021,
        COUNT(*)                    AS n_sa2,
        ROUND(SUM(area_sqkm), 1)    AS total_sqkm
    FROM qld_sa2_enriched
    GROUP BY sa4_name_2021
    ORDER BY total_sqkm DESC
"""))

## 7. Write the enriched result back to a `.tab`

`df.write.format("geo")` writes a new MapInfo `.tab` (with the computed `area_sqkm`
and centroid columns) back into the Volume. The writer reads the `geometry` column
(WKT strings) as the feature geometry and persists the rest as attributes; `crs`
declares the output coordinate system.

In [ ]:
output_tab = f"{volume_path}/qld_sa2_enriched.tab"

# The writer expects a 'geometry' column holding WKT. Keep it, add computed columns.
df_to_write = df_enriched.select(
    "geometry",
    "sa2_code_2021", "sa2_name_2021", "sa3_name_2021", "sa4_name_2021",
    "gccsa_name_2021", "state_name_2021",
    "area_sqkm", "centroid_lon", "centroid_lat",
)

(
    df_to_write.write.format("geo")
    .option("path", output_tab)
    .option("crs", "EPSG:4326")
    .mode("overwrite")
    .save()
)

print(f"Wrote enriched .tab: {output_tab}")
print("Output sidecar files:")
for f in sorted(os.listdir(volume_path)):
    if f.startswith("qld_sa2_enriched"):
        print("  ", f)

## 8. Verify the round-trip

Read the freshly written `.tab` back through the custom source and confirm the
feature count matches and the new `area_sqkm` column is present.

In [ ]:
df_verify = (
    spark.read.format("geo")
    .option("path", output_tab)
    .load()
)

n_in = df_sa2.count()
n_out = df_verify.count()
has_area = "area_sqkm" in df_verify.columns

print(f"Input  .tab features : {n_in}")
print(f"Output .tab features : {n_out}")
print(f"'area_sqkm' present  : {has_area}")
assert n_out == n_in, f"Feature count mismatch: {n_out} != {n_in}"
assert has_area, "area_sqkm column missing from output .tab"
print("\n✅ ROUND-TRIP VERIFIED — read .tab → Spatial SQL → write .tab, feature counts match.")

display(
    df_verify.select("sa2_name_2021", "sa4_name_2021", "area_sqkm", "centroid_lon", "centroid_lat")
    .orderBy("area_sqkm", ascending=False)
    .limit(10)
)

## Summary

| Step | What happened |
|------|----------------|
| Source | ABS ASGS 2021 SA2 (Greater Brisbane, QLD) → seeded MapInfo `.tab` in a UC Volume |
| Read | `spark.read.format("geo")` read the `.tab` (geometry as WKT + attributes) |
| Process | Spatial SQL `ST_Transform` (EPSG:3577) + `ST_Area` → `area_sqkm`; `ST_Centroid` |
| Write | `df.write.format("geo")` wrote the enriched result back to a new `.tab` |
| Verify | Re-read the output `.tab`; feature count matches and `area_sqkm` present |

The custom `geo` data source now supports **both reading and writing** MapInfo `.tab`
(and Shapefile, GeoJSON, GeoPackage, KML), runs on **serverless**, and interoperates
with native Databricks **Spatial SQL** — no Mosaic/Sedona required.

### A note on scale
MapInfo `.tab` is a **single-file format** (`.tab`/`.dat`/`.map`/`.id` read through one
GDAL handle), so it is inherently **not parallel** — the custom `geo` source is best used
as an **ingestion/egress bridge** for `.tab`, not as a query engine. The write path here
is already distributed (each Spark partition serializes in parallel; only the final
single-file merge is on the driver, which a `.tab` requires).

For large or repeated spatial workloads, the optimised pattern is: **read the `.tab`
once** with this source, **land it as Delta with a native `GEOMETRY` column** (store as
WKB via `ST_AsBinary`, rehydrate with `ST_GeomFromWKB`), then run all Spatial SQL on
Delta. That is where Databricks parallelises properly — Photon, spatial predicate
pushdown, and H3 indexing — and is the real advantage over desktop MapInfo. Write back
to `.tab` only at the end if the customer needs a MapInfo deliverable.

## Clean up (optional)
Uncomment to remove the demo files from the Volume.

In [ ]:
# for f in os.listdir(volume_path):
#     if f.startswith("qld_sa2_"):
#         os.remove(os.path.join(volume_path, f))
# print("Cleaned up demo .tab files.")